In [1]:
import sqlite3

def connect():
    return sqlite3.connect("finance.db")

def create_tables():
    conn = connect()
    cursor = conn.cursor()

    # Users table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE,
        password TEXT
    )
    """)

    # Transactions table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS transactions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        amount REAL,
        category TEXT,
        type TEXT,
        date TEXT
    )
    """)

    # Budget table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS budgets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        category TEXT,
        limit_amount REAL
    )
    """)

    conn.commit()
    conn.close()

create_tables()
print("✅ Step 1 Done")

✅ Step 1 Done


In [2]:
def register():
    username = input("Enter username: ")
    password = input("Enter password: ")

    conn = connect()
    cursor = conn.cursor()

    try:
        cursor.execute(
            "INSERT INTO users (username, password) VALUES (?, ?)",
            (username, password)
        )
        conn.commit()
        print("✅ Registered successfully!")
    except:
        print("❌ Username already exists!")

    conn.close()


def login():
    username = input("Enter username: ")
    password = input("Enter password: ")

    conn = connect()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT id FROM users WHERE username=? AND password=?",
        (username, password)
    )

    user = cursor.fetchone()
    conn.close()

    if user:
        print("✅ Login successful!")
        return user[0]
    else:
        print("❌ Invalid credentials!")
        return None

In [3]:
def add_transaction(user_id):
    amount = float(input("Enter amount: "))
    category = input("Enter category: ")
    t_type = input("Enter type (income/expense): ")
    date = input("Enter date (YYYY-MM-DD): ")

    conn = connect()
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO transactions (user_id, amount, category, type, date)
        VALUES (?, ?, ?, ?, ?)
    """, (user_id, amount, category, t_type, date))

    conn.commit()
    conn.close()

    print("✅ Transaction added successfully!")

    check_budget(user_id, category)

In [4]:
def view_transactions(user_id):
    conn = connect()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT amount, category, type, date
        FROM transactions
        WHERE user_id=?
    """, (user_id,))

    rows = cursor.fetchall()
    conn.close()

    print("\n--- Your Transactions ---")
    for row in rows:
        print(f"Amount: {row[0]}, Category: {row[1]}, Type: {row[2]}, Date: {row[3]}")

In [5]:
def monthly_report(user_id):
    month = input("Enter month (MM): ")

    conn = connect()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT type, SUM(amount)
        FROM transactions
        WHERE user_id=? AND strftime('%m', date)=?
        GROUP BY type
    """, (user_id, month))

    data = cursor.fetchall()
    conn.close()

    income = 0
    expense = 0

    for row in data:
        if row[0] == "income":
            income = row[1]
        else:
            expense = row[1]

    print("\n--- Monthly Report ---")
    print(f"Total Income: {income}")
    print(f"Total Expense: {expense}")
    print(f"Savings: {income - expense}")

In [6]:
def set_budget(user_id):
    category = input("Enter category: ")
    limit_amount = float(input("Enter budget limit: "))

    conn = connect()
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO budgets (user_id, category, limit_amount)
        VALUES (?, ?, ?)
    """, (user_id, category, limit_amount))

    conn.commit()
    conn.close()

    print("✅ Budget set!")


def check_budget(user_id, category):
    conn = connect()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT limit_amount FROM budgets
        WHERE user_id=? AND category=?
    """, (user_id, category))

    result = cursor.fetchone()

    if result:
        limit_amount = result[0]

        cursor.execute("""
            SELECT SUM(amount) FROM transactions
            WHERE user_id=? AND category=? AND type='expense'
        """, (user_id, category))

        spent = cursor.fetchone()[0]

        if spent and spent > limit_amount:
            print("⚠️ Budget exceeded!")

    conn.close()

In [12]:
def main():
    while True:
        print("\n--- Personal Finance App ---")
        print("1. Register")
        print("2. Login")
        print("3. Exit")

        choice = input("Enter choice: ")

        if choice == "1":
            register()

        elif choice == "2":
            user_id = login()

            if user_id:
                while True:   # 🔥 THIS LOOP IS IMPORTANT
                    print("\n--- Dashboard ---")
                    print("1. Add Transaction")
                    print("2. View Transactions")
                    print("3. Monthly Report")
                    print("4. Set Budget")
                    print("5. Backup Data")
                    print("6. Restore Data")
                    print("7. Logout")

                    option = input("Enter choice: ")

                    if option == "1":
                        add_transaction(user_id)

                    elif option == "2":
                        view_transactions(user_id)

                    elif option == "3":
                        monthly_report(user_id)

                    elif option == "4":
                        set_budget(user_id)

                    elif option == "5":
                        backup_data()

                    elif option == "6":
                        restore_data()

                    elif option == "7":
                        print("Logged out!")
                        break   # ✅ NOW inside loop (correct)

                    else:
                        print("Invalid choice!")

        elif choice == "3":
            print("Goodbye!")
            break

        else:
            print("Invalid choice!")

main()


--- Personal Finance App ---
1. Register
2. Login
3. Exit


Enter choice:  2
Enter username:  vv
Enter password:  123


✅ Login successful!

--- Dashboard ---
1. Add Transaction
2. View Transactions
3. Monthly Report
4. Set Budget
5. Backup Data
6. Restore Data
7. Logout


Enter choice:  1
Enter amount:  100
Enter category:  food
Enter type (income/expense):  expense
Enter date (YYYY-MM-DD):  2026-05-27


✅ Transaction added successfully!

--- Dashboard ---
1. Add Transaction
2. View Transactions
3. Monthly Report
4. Set Budget
5. Backup Data
6. Restore Data
7. Logout


...

✅ Data restored successfully!

--- Dashboard ---
1. Add Transaction
2. View Transactions
3. Monthly Report
4. Set Budget
5. Backup Data
6. Restore Data
7. Logout


Enter choice:  7


Logged out!

--- Personal Finance App ---
1. Register
2. Login
3. Exit


Enter choice:  3


Goodbye!


In [13]:
import shutil

def backup_data():
    try:
        shutil.copy("finance.db", "backup.db")
        print("✅ Backup created successfully!")
    except:
        print("❌ Backup failed!")


def restore_data():
    try:
        shutil.copy("backup.db", "finance.db")
        print("✅ Data restored successfully!")
    except:
        print("❌ No backup found!")